In [12]:
import pandas as pd
customers = pd.DataFrame({
    "customer_id": ["C01", "C02", "C03", "C04"],
    "signup_date": ["2025-11-01", "2026-01-15", "2026-02-10", "2026-02-20"],
    "segment": ["new", "returning", "new", "business"],
})

orders = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5, 6],
    "customer_id": ["C01", "C01", "C02", "C03", "C03", "C05"],
    "order_date": [
        "2025-12-01", "2026-01-10", "2026-02-01",
        "2026-02-15", "2026-03-01", "2026-02-20",
    ],
    "amount": [100, 150, 200, 50, 75, 500],
    "status": ["complete", "complete", "refunded", "complete", "complete", "complete"],
})

In [13]:
def customer_lifetime_pipeline(customers,orders):
    customers=customers.copy()
    orders=orders.copy()

    customers["signup_date"]=pd.to_datetime(customers["signup_date"],errors="coerce")
    orders["order_date"]=pd.to_datetime(orders["order_date"],errors="coerce")

    valid_orders=orders[orders["status"].str.lower()!="refunded"].copy()

    valid_customers=customers.dropna(subset=['customer_id','signup_date'])
    valid_orders=valid_orders.dropna(subset=['order_id','customer_id','order_date','amount'])
    valid_orders=valid_orders[valid_orders["amount"]>0]

    unmatched_orders=valid_orders[
        ~valid_orders["customer_id"].isin(valid_customers["customer_id"])
    ]
    matched_orders=valid_orders[
        valid_orders["customer_id"].isin(valid_customers["customer_id"])
    ]

    customers_stats=matched_orders.groupby('customer_id').agg(
        total_orders=('order_id','nunique'),
        total_revenue=('amount','sum'),
        average_order_value=('amount','mean'),
        first_order_date=('order_date','min'),
    ).reset_index()

    customer_summary=valid_customers.merge(customers_stats,on='customer_id',how='left').fillna({
        "total_orders":0,
        "total_revenue":0,
        "average_order_value":0
    })

    customer_summary["days_from_signup_to_first_order"]=(
        customer_summary["singup_date"]-customer_summary["first_order_date"]
    ).dt.days
    print(customer_summary)


In [14]:
customer_lifetime_pipeline(customers,orders)

KeyError: 'singup_date'